# NT120 — Pandas: làm sạch và tái lập kết quả

Nội dung tách từ Lecture 03 — slides 22, 59, 60, 63–66, 70, 71, 75, 83.

Notebook tự tạo dữ liệu mô phỏng và chạy độc lập từ trên xuống.

In [1]:
# Slide 37 — Standard header
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 40)

# Works whether the notebook is opened from the project root or notebooks/.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "requirements.txt").exists()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
FLOW_PATH = DATA_DIR / "flows.csv"
print("pandas:", pd.__version__, "| numpy:", np.__version__)

pandas: 2.3.3 | numpy: 2.2.6


In [2]:
# Dữ liệu nhỏ mô phỏng cho các ví dụ Pandas bên dưới
raw = pd.DataFrame({
    "dst_port": [22, 80, 443, 8080, 53, 22, 80, 3389],
    "duration": [3.395, 61.5, 0.04, 120.0, 0.0, 4.1, 0.9, 0.02],
    "fwd_pkts": [8, 12, 1, 20, 2, 6, 3, 1],
    "bwd_pkts": [7, 10, 0, 19, 1, 4, 2, 0],
    "fwd_bytes": [6520, 22000, 0, 402110, 95, 7047, 250, 60],
    "bwd_bytes": ["6,520", "22,000", "0", "402,110", "95", "7,047", "250", "60"],
    "protocol": ["TCP", "UDP", "TCP", "TCP", "UDP", "TCP", "UDP", "TCP"],
    "label": ["BENIGN", "BENIGN", "PortScan", "BENIGN", "BENIGN", "BENIGN", "BENIGN", "PortScan"],
    "ttl": [64, "unknown", 128, 64, 255, 64, 64, 128],
    "start": pd.date_range("2026-09-14", periods=8, freq="D"),
})
raw.to_csv(FLOW_PATH, index=False)
raw.head()

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start
0,22,3.395,8,7,6520,"6,520",TCP,BENIGN,64,2026-09-14
1,80,61.500,12,10,22000,"22,000",UDP,BENIGN,unknown,2026-09-15
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16
3,8080,120.000,20,19,402110,"402,110",TCP,BENIGN,64,2026-09-17
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18


In [3]:
# Load the teaching table for the transformations below
df = pd.read_csv(FLOW_PATH, thousands=",")
df.head()

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18


In [4]:
# Slide 22 — validation function
def validate(df):
    text = set(df.columns) - set(df.select_dtypes("number").columns)
    assert text == {"protocol", "label", "start", "ttl"}, (
        f"unexpected text columns: {sorted(text)}"
    )
    assert set(df["label"]) <= {"BENIGN", "PortScan"}
    assert df["dst_port"].between(0, 65535).all()
    assert (df["duration"] >= 0).all()
    assert not df.duplicated().any()
    return df

# Bản slide gốc dùng schema chỉ gồm protocol và label là cột text.
# Dữ liệu mô phỏng thêm start/ttl để minh hoạ các slide sau.

## Join, cột dẫn xuất và làm sạch — slides 59, 60, 63–66

In [5]:
# Slide 59 — left join with key validation
services = pd.DataFrame({"dst_port": [22, 80, 443], "service": ["ssh", "http", "https"]})
n = len(df)
m = df.merge(services, on="dst_port", how="left", validate="many_to_one")
assert len(m) == n
m["service"] = m["service"].fillna("other")

bad = pd.DataFrame({"dst_port": [22, 80, 80, 443], "service": ["ssh", "http", "web", "https"]})
print("rows without validation:", len(df.merge(bad, on="dst_port", how="left")))
try:
    df.merge(bad, on="dst_port", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print(type(error).__name__, "— duplicate lookup key detected")
m.head()

rows without validation: 10
MergeError — duplicate lookup key detected


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,service
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,ssh
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,http
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,https
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,other
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,other


In [6]:
# Slide 60 — derived columns
df = df.assign(
    bytes_total=df["fwd_bytes"] + df["bwd_bytes"],
    bytes_per_pkt=lambda d: d["bytes_total"] / (d["fwd_pkts"] + d["bwd_pkts"]),
    is_web=df["dst_port"].isin([80, 443, 8080]),
    dur_class=pd.cut(df["duration"], [-0.001, 0.1, 10, 121], labels=["short", "medium", "long"]),
)
display(df["dur_class"].value_counts())
display(df.sort_values("duration", ascending=False).head(3))
df.nlargest(3, "fwd_bytes")

dur_class
short     3
medium    3
long      2
Name: count, dtype: int64

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,bytes_total,bytes_per_pkt,is_web,dur_class
3,8080,120.0,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,804220,20621.025641,True,long
1,80,61.5,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,44000,2000.000000,True,long
5,22,4.1,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,14094,1409.400000,False,medium


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,bytes_total,bytes_per_pkt,is_web,dur_class
3,8080,120.0,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,804220,20621.025641,True,long
1,80,61.5,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,44000,2000.000000,True,long
5,22,4.1,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,14094,1409.400000,False,medium


In [7]:
# Slide 63 — convert text safely
small = raw[["ttl"]].copy()
before = small["ttl"].isna().sum()
small["ttl"] = pd.to_numeric(small["ttl"], errors="coerce")
lost = small["ttl"].isna().sum() - before
print(lost, "values could not be parsed")

df["bwd_bytes"] = pd.to_numeric(df["bwd_bytes"], errors="raise")
print(df["bwd_bytes"].dtype)
df["bwd_bytes"].describe()

1 values could not be parsed
int64


count         8.000000
mean      54760.250000
std      140549.542863
min           0.000000
25%          86.250000
50%        3385.000000
75%       10785.250000
max      402110.000000
Name: bwd_bytes, dtype: float64

In [8]:
# Slides 64–65 — duplicates, outliers, encoding, scaling
print("duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("negative durations:", (df["duration"] < 0).sum())
assert df["dst_port"].between(0, 65535).all()

train = df.sample(frac=0.75, random_state=RANDOM_STATE)
mu, sd = train["duration"].mean(), train["duration"].std()
outlier = (df["duration"] - mu).abs() > 3 * sd
print("outlier share:", outlier.mean())

display(pd.get_dummies(df["protocol"], prefix="proto"))
display(pd.cut(df["dst_port"], [-1, 1023, 49151, 65535], labels=["well-known", "registered", "dynamic"]))
z = (df["duration"] - mu) / sd
vocab = sorted(train["protocol"].unique())
print("vocabulary from training:", vocab)

duplicate rows: 0
negative durations: 0
outlier share: 0.125


,proto_TCP,proto_UDP
0,True,False
1,False,True
2,True,False
3,True,False
4,False,True
5,True,False
6,False,True
7,True,False


0    well-known
1    well-known
2    well-known
3    registered
4    well-known
5    well-known
6    well-known
7    registered
Name: dst_port, dtype: category
Categories (3, object): ['well-known' < 'registered' < 'dynamic']

vocabulary from training: ['TCP', 'UDP']


In [9]:
# Slide 66 — raw in, clean and validated out
def clean(raw):
    df = raw.copy()
    df["bwd_bytes"] = pd.to_numeric(
        df["bwd_bytes"].astype(str).str.replace(",", "")
    )
    return df.drop_duplicates()

raw = pd.read_csv(FLOW_PATH)
df_clean = validate(clean(raw))
df_clean.dtypes

dst_port       int64
duration     float64
fwd_pkts       int64
bwd_pkts       int64
fwd_bytes      int64
bwd_bytes      int64
protocol      object
label         object
ttl           object
start         object
dtype: object

## Tái lập kết quả và checkpoint — slides 70, 71 và 75

In [10]:
# Slide 70 — seeded randomness and a fixed reference date
rng = np.random.default_rng(42)
print(rng.normal(size=3))
print(df_clean.sample(frac=0.1, random_state=42))
df_clean["start"] = pd.to_datetime(df_clean["start"], errors="raise")
df_clean["start"] = pd.to_datetime(df_clean["start"], errors="raise")
REF = pd.Timestamp("2026-09-21")
df_clean["age_days"] = (REF - df_clean["start"]).dt.days
df_clean[["start", "age_days"]].head()

[ 0.30471708 -1.03998411  0.7504512 ]
   dst_port  duration  fwd_pkts  bwd_pkts  fwd_bytes  bwd_bytes protocol  \
1        80      61.5        12        10      22000      22000      UDP   

    label      ttl       start  
1  BENIGN  unknown  2026-09-15  


,start,age_days
0,2026-09-14,7
1,2026-09-15,6
2,2026-09-16,5
3,2026-09-17,4
4,2026-09-18,3


In [11]:
# Slide 71 — an idempotent transformation
# Unsafe toggle shown in the slide (do not run twice):
# df["duration"] = df["duration"] / 60

# Safe switch: derive from raw values; rerunning returns the same answer.
df_clean["duration_min"] = raw["duration"] / 60
df_clean["bwd_bytes"] = pd.to_numeric(
    df_clean["bwd_bytes"].astype(str).str.replace(",", "")
)
df_clean[["duration", "duration_min", "bwd_bytes"]].head()

,duration,duration_min,bwd_bytes
0,3.395,0.056583,6520
1,61.500,1.025000,22000
2,0.040,0.000667,0
3,120.000,2.000000,402110
4,0.000,0.000000,95


In [12]:
# Slide 75 — checkpoint
checkpoint = pd.DataFrame({"port": [22, 80, 22, 443], "bytes": [100, 250, 90, 500]})
print(checkpoint.shape)
print(checkpoint[checkpoint["port"] == 22]["bytes"].mean())
print(checkpoint.groupby("port")["bytes"].sum().loc[22])

(4, 2)
95.0
190


In [13]:
# Slide 83 — Pandas: transform, group, combine and save
# Cell khởi tạo: chạy cell này trước các cell tham khảo bên dưới.
ref = df_clean.copy()
s = pd.Series(["6,520", "unknown", "  TCP "], name="example")

0    6520.0
1       NaN
2       NaN
Name: example, dtype: float64

0       6520
1    unknown
2        tcp
Name: example, dtype: object

,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,age_days,duration_min
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,7,0.056583
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,6,1.025000
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,5,0.000667
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,4,2.000000
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,3,0.000000
5,22,4.100,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,2,0.068333
6,80,0.900,3,2,250,250,UDP,BENIGN,64,2026-09-20,1,0.015000
7,3389,0.020,1,0,60,60,TCP,PortScan,128,2026-09-21,0,0.000333


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,age_days,duration_min
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,7,0.056583
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,6,1.025000
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,5,0.000667
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,4,2.000000
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,3,0.000000
5,22,4.100,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,2,0.068333
6,80,0.900,3,2,250,250,UDP,BENIGN,64,2026-09-20,1,0.015000
7,3389,0.020,1,0,60,60,TCP,PortScan,128,2026-09-21,0,0.000333


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,age_days,duration_min,rate,duration_band,duration_quartile
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,7,0.056583,1920.471281,"(-0.12, 40.0]","(2.148, 18.45]"
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,6,1.025000,357.723577,"(40.0, 80.0]","(18.45, 120.0]"
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,5,0.000667,0.000000,"(-0.12, 40.0]","(0.035, 2.148]"
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,4,2.000000,3350.916667,"(80.0, 120.0]","(18.45, 120.0]"
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,3,0.000000,0.000000,"(-0.12, 40.0]","(-0.001, 0.035]"
5,22,4.100,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,2,0.068333,1718.780488,"(-0.12, 40.0]","(2.148, 18.45]"
6,80,0.900,3,2,250,250,UDP,BENIGN,64,2026-09-20,1,0.015000,277.777778,"(-0.12, 40.0]","(0.035, 2.148]"
7,3389,0.020,1,0,60,60,TCP,PortScan,128,2026-09-21,0,0.000333,3000.000000,"(-0.12, 40.0]","(-0.001, 0.035]"


...

label,BENIGN,PortScan
protocol,,
TCP,3,2
UDP,3,0


label,BENIGN,PortScan
protocol,,
TCP,42.498333,0.03
UDP,20.800000,NaN


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,age_days,duration_min,name
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,7,0.056583,ssh
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,6,1.025000,http
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,5,0.000667,https
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,4,2.000000,NaN
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,3,0.000000,NaN
5,22,4.100,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,2,0.068333,ssh
6,80,0.900,3,2,250,250,UDP,BENIGN,64,2026-09-20,1,0.015000,http
7,3389,0.020,1,0,60,60,TCP,PortScan,128,2026-09-21,0,0.000333,NaN


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,age_days,duration_min
3,8080,120.000,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,4,2.000000
1,80,61.500,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,6,1.025000
5,22,4.100,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,2,0.068333
0,22,3.395,8,7,6520,6520,TCP,BENIGN,64,2026-09-14,7,0.056583
6,80,0.900,3,2,250,250,UDP,BENIGN,64,2026-09-20,1,0.015000
2,443,0.040,1,0,0,0,TCP,PortScan,128,2026-09-16,5,0.000667
7,3389,0.020,1,0,60,60,TCP,PortScan,128,2026-09-21,0,0.000333
4,53,0.000,2,1,95,95,UDP,BENIGN,255,2026-09-18,3,0.000000


,dst_port,duration,fwd_pkts,bwd_pkts,fwd_bytes,bwd_bytes,protocol,label,ttl,start,age_days,duration_min
3,8080,120.0,20,19,402110,402110,TCP,BENIGN,64,2026-09-17,4,2.000000
1,80,61.5,12,10,22000,22000,UDP,BENIGN,unknown,2026-09-15,6,1.025000
5,22,4.1,6,4,7047,7047,TCP,BENIGN,64,2026-09-19,2,0.068333


In [ ]:
# Chuyển text thành số; giá trị không chuyển được thành NaN
numeric_s = pd.to_numeric(s.str.replace(",", ""), errors="coerce")
display(numeric_s)

In [ ]:
# Làm sạch text: bỏ dấu phẩy, đổi chữ thường, bỏ khoảng trắng
clean_text = s.str.replace(",", "").str.lower().str.strip()
display(clean_text)

In [ ]:
# Điền giá trị thiếu bằng 0
display(ref.fillna(0))

In [ ]:
# Bỏ các hàng thiếu duration
display(ref.dropna(subset=["duration"]))

In [ ]:
# Thêm các cột biến đổi
transformed = ref.assign(
    rate=np.where(ref["duration"] > 0, ref["fwd_bytes"] / ref["duration"], 0.0),
    duration_band=pd.cut(ref["duration"], bins=3),
    duration_quartile=pd.qcut(ref["duration"], 4, duplicates="drop"),
)
display(transformed)

In [ ]:
# Group và aggregate
display(ref.groupby("protocol")["duration"].agg(["mean", "median", "size"]))

In [ ]:
# Bảng chéo protocol × label
display(pd.crosstab(ref["protocol"], ref["label"]))

In [ ]:
# Pivot table: duration trung bình theo protocol và label
display(ref.pivot_table(index="protocol", columns="label", values="duration", aggfunc="mean"))

In [ ]:
# Join thêm tên dịch vụ theo port
lookup = pd.DataFrame({"dst_port": [22, 80, 443], "name": ["ssh", "http", "https"]})
display(ref.merge(lookup, on="dst_port", how="left", validate="many_to_one"))

In [ ]:
# Sắp xếp duration giảm dần
display(ref.sort_values("duration", ascending=False))

In [ ]:
# Ba flow có fwd_bytes lớn nhất
display(ref.nlargest(3, "fwd_bytes"))

In [ ]:
# Lưu dữ liệu (bỏ comment khi cần)
# ref.to_csv("out.csv", index=False)
# ref.to_parquet("out.parquet")  # requires pyarrow or fastparquet